In [1]:
# Système et fichiers
import sqlite3
import warnings
from pathlib import Path

# Manipulation de données
import numpy as np
import pandas as pd

# Distances géographiques
from sklearn.neighbors import BallTree

warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.max_columns", None)


# Base de données relationnelle IRVE

La base SQLite `data/database/irve_database.db` est construite à partir des deux jeux propres produits par `01_eda_cleaning.ipynb` : `irve_points_clean.csv` (un point de charge par ligne) et `communes_population_clean.csv` (une commune par ligne, y compris les communes sans borne). Les noms de départements et de régions viennent du référentiel communal 2025.

## Modèle conceptuel (MCD)

```mermaid
erDiagram
    REGION ||--|{ DEPARTEMENT : "regroupe"
    DEPARTEMENT ||--|{ COMMUNE : "regroupe"
    COMMUNE ||--o{ STATION : "accueille"
    ACTEUR ||--o{ STATION : "amenage"
    ACTEUR ||--o{ STATION : "exploite"
    STATION ||--|{ POINT_DE_CHARGE : "comporte"

    REGION {
        text code_region PK
        text nom_region
    }
    DEPARTEMENT {
        text code_departement PK
        text nom_departement
        text code_region FK
    }
    COMMUNE {
        text code_insee PK
        text nom_commune
        int population
        real latitude
        real longitude
        real distance_pdc_km
        real distance_150kw_km
        int desert_recharge
        int desert_150kw
        text code_departement FK
    }
    ACTEUR {
        int id_acteur PK
        text nom_acteur
    }
    STATION {
        text id_station PK
        text nom_station
        text adresse_station
        text implantation_station
        text condition_acces
        real latitude
        real longitude
        text code_insee FK
        int id_amenageur FK
        int id_operateur FK
    }
    POINT_DE_CHARGE {
        text id_pdc PK
        real puissance_kw
        int prise_type_2
        int prise_type_combo_ccs
        int prise_type_chademo
        int prise_type_ef
        text date_mise_en_service
        text id_station FK
    }
```

| Association | Cardinalités | Lecture |
|---|---|---|
| REGION – DEPARTEMENT | 1,n – 1,1 | Une région regroupe au moins un département ; un département appartient à une seule région |
| DEPARTEMENT – COMMUNE | 1,n – 1,1 | Un département regroupe au moins une commune ; une commune appartient à un seul département |
| COMMUNE – STATION | 0,n – 1,1 | Une commune peut n'avoir **aucune** station (désert) ; une station est située dans une seule commune |
| ACTEUR – STATION (aménage) | 0,n – 1,1 | Un acteur aménage zéro ou plusieurs stations ; une station a un seul aménageur |
| ACTEUR – STATION (exploite) | 0,n – 1,1 | Un acteur exploite zéro ou plusieurs stations ; une station a un seul opérateur |
| STATION – POINT_DE_CHARGE | 1,n – 1,1 | Une station comporte au moins un point de charge ; un point appartient à une seule station |

## Modèle logique (MLD)

- **REGION** (<u>code_region</u>, nom_region)
- **DEPARTEMENT** (<u>code_departement</u>, nom_departement, #code_region)
- **COMMUNE** (<u>code_insee</u>, nom_commune, population, latitude, longitude, distance_pdc_km, distance_150kw_km, desert_recharge, desert_150kw, #code_departement)
- **ACTEUR** (<u>id_acteur</u>, nom_acteur)
- **STATION** (<u>id_station</u>, nom_station, adresse_station, implantation_station, condition_acces, latitude, longitude, #code_insee, #id_amenageur, #id_operateur)
- **POINT_DE_CHARGE** (<u>id_pdc</u>, puissance_kw, prise_type_2, prise_type_combo_ccs, prise_type_chademo, prise_type_ef, date_mise_en_service, #id_station)
- Vue **v_equipement_commune** : nombre de stations, de points et de points de 150 kW ou plus par commune, recalculé à partir des tables.

## Justification du schéma

| Choix | Justification |
|---|---|
| Région et département dans des tables séparées | La région dépend du département, pas de la commune : la stocker dans COMMUNE créerait une dépendance transitive (contraire à la 3e forme normale) et répéterait le même nom de région sur chaque commune |
| Une seule table ACTEUR, référencée deux fois par STATION | L'aménageur (qui finance) et l'opérateur (qui exploite) sont deux rôles distincts : ils diffèrent pour 62 % des points. Une même entreprise peut tenir l'un ou l'autre rôle selon la station ; son nom n'est stocké qu'une fois |
| Attributs répartis entre STATION et POINT_DE_CHARGE | Nom, adresse, implantation, accès, acteurs et position sont identiques pour 98,4 % à 99,96 % des stations : ils dépendent de la station. La puissance varie dans 34,9 % des stations et la date dans 4,4 % : elles dépendent du point de charge |
| Cardinalité 0,n entre COMMUNE et STATION | Les communes sans borne restent dans la base : c'est ce qui permet de mesurer les déserts de recharge |
| Distances et indicateurs de désert stockés dans COMMUNE | Calculés en Python (distance géographique), ils ne peuvent pas être recalculés en SQL standard |
| Comptages dans une vue, pas dans une table | Le nombre de points par commune se déduit des tables : le stocker créerait une redondance qui pourrait devenir incohérente |
| Contraintes CHECK, NOT NULL et clés étrangères | La base refuse une puissance nulle ou supérieure à 600 kW, une prise différente de 0 ou 1, une station sans commune ou un point sans station |

## Règles de chargement

| Cas | Règle | Effet |
|---|---|---|
| Station dont les points sont dans plusieurs communes | Une station par commune : identifiant suffixé par le code INSEE | Une station appartient à une seule commune |
| Attribut de station différent selon ses points | Valeur la plus fréquente parmi les points de la station | Une seule valeur par station |
| Commune des points absente de la table communale (population nulle) | Commune ajoutée avec sa population réelle de 0 | Aucun point de charge perdu |
| `date_mise_en_service` absente | Laissée vide (NULL) | Seule colonne facultative : non publiée par plusieurs producteurs |


In [2]:
# ===================================================================
# PRÉPARATION DES TABLES À PARTIR DES DEUX JEUX PROPRES
# ===================================================================
CHEMIN_POINTS = Path("../data/processed/irve_points_clean.csv")
CHEMIN_COMMUNES = Path("../data/processed/communes_population_clean.csv")
CHEMIN_REFERENTIEL = Path("../data/raw/referentiel_communes.csv")
RAYON_TERRE_KM = 6371.0088
PUISSANCE_150KW = 150

journal = []


def journaliser(etape, avant, apres, concernees, detail):
  journal.append({"etape": etape, "lignes_avant": avant, "lignes_apres": apres,
                  "lignes_concernees": int(concernees), "detail": detail})
  print(f"[{etape}] {avant:,} -> {apres:,} lignes | concernées : {int(concernees):,} | {detail}")


points = pd.read_csv(CHEMIN_POINTS, dtype=str)
communes = pd.read_csv(CHEMIN_COMMUNES, dtype={"code_insee": str, "code_departement": str})
referentiel = pd.read_csv(CHEMIN_REFERENTIEL, sep=";", dtype=str,
                          usecols=["com_code", "com_name", "dep_code", "dep_name",
                                   "reg_code", "reg_name", "geo_point_2d"])
for colonne in ["puissance_kw", "latitude", "longitude"]:
  points[colonne] = points[colonne].astype(float)
journaliser("1_lecture_points", len(points), len(points), 0, "points de charge du jeu propre")
journaliser("1_lecture_communes", len(communes), len(communes), 0, "communes du jeu propre")

# --- Commune des points absente de la table communale (population nulle) ---
absentes = sorted(set(points["code_insee"]) - set(communes["code_insee"]))
ajout = referentiel.loc[referentiel["com_code"].isin(absentes)].copy()
ajout[["latitude", "longitude"]] = (
    ajout["geo_point_2d"].str.split(",", expand=True).astype(float).to_numpy())


def distance_plus_proche_km(depuis, vers):
  arbre = BallTree(np.radians(vers[["latitude", "longitude"]].to_numpy()), metric="haversine")
  distance, _ = arbre.query(np.radians(depuis[["latitude", "longitude"]].to_numpy()), k=1)
  return (distance[:, 0] * RAYON_TERRE_KM).round(2)


ajout = ajout.rename(columns={"com_code": "code_insee", "com_name": "nom_commune",
                              "dep_code": "code_departement", "reg_name": "region"})
ajout["population"] = 0
ajout["distance_pdc_km"] = distance_plus_proche_km(ajout, points)
ajout["distance_150kw_km"] = distance_plus_proche_km(
    ajout, points.loc[points["puissance_kw"] >= PUISSANCE_150KW])
ajout["desert_recharge"] = ajout["distance_pdc_km"] > 10
ajout["desert_150kw"] = ajout["distance_150kw_km"] > 30
avant = len(communes)
communes = pd.concat([communes, ajout[communes.columns.drop(
    ["nb_pdc", "nb_stations", "nb_pdc_150kw"])]], ignore_index=True)
journaliser("2_commune_population_nulle", avant, len(communes), len(ajout),
            "commune des points ajoutée avec sa population réelle de 0 : "
            + ", ".join(ajout["nom_commune"]))

# --- Régions et départements ---------------------------------------
geographie = referentiel.loc[referentiel["com_code"].isin(communes["code_insee"])]
table_region = (geographie[["reg_code", "reg_name"]].drop_duplicates()
                .rename(columns={"reg_code": "code_region", "reg_name": "nom_region"})
                .sort_values("code_region").reset_index(drop=True))
table_departement = (geographie[["dep_code", "dep_name", "reg_code"]].drop_duplicates()
                     .rename(columns={"dep_code": "code_departement", "dep_name": "nom_departement",
                                      "reg_code": "code_region"})
                     .sort_values("code_departement").reset_index(drop=True))
journaliser("3_regions", len(table_region), len(table_region), 0, "régions")
journaliser("3_departements", len(table_departement), len(table_departement), 0, "départements")

table_commune = communes[[
    "code_insee", "nom_commune", "population", "latitude", "longitude", "distance_pdc_km",
    "distance_150kw_km", "desert_recharge", "desert_150kw", "code_departement",
]].copy()
table_commune[["desert_recharge", "desert_150kw"]] = (
    table_commune[["desert_recharge", "desert_150kw"]].astype(int))

# --- Acteurs : aménageurs et opérateurs dans une même table -------
noms_acteurs = sorted(set(points["nom_amenageur"]) | set(points["nom_operateur"]))
table_acteur = pd.DataFrame({"id_acteur": range(1, len(noms_acteurs) + 1),
                             "nom_acteur": noms_acteurs})
id_acteur = table_acteur.set_index("nom_acteur")["id_acteur"]
journaliser("4_acteurs", len(table_acteur), len(table_acteur),
            (points["nom_amenageur"] == points["nom_operateur"]).sum(),
            "acteurs distincts ; concernées = points dont l'aménageur est aussi l'opérateur")

# --- Stations : une station par commune ----------------------------
communes_par_station = points.groupby("id_station")["code_insee"].transform("nunique")
scindee = communes_par_station > 1
points.loc[scindee, "id_station"] = (
    points.loc[scindee, "id_station"] + "_" + points.loc[scindee, "code_insee"])
journaliser("5_stations_scindees", len(points), len(points), scindee.sum(),
            f"points de {points.loc[scindee, 'id_station'].str.rsplit('_', n=1).str[0].nunique()}"
            " stations réparties sur plusieurs communes : une station par commune")

# Attribut de station : valeur la plus fréquente parmi ses points
points["position"] = points["latitude"].astype(str) + "|" + points["longitude"].astype(str)
attributs_station = ["nom_station", "adresse_station", "implantation_station",
                     "condition_acces", "nom_amenageur", "nom_operateur", "position", "code_insee"]
groupes = points.groupby("id_station")
desaccord = (groupes[attributs_station].nunique() > 1).any(axis=1)
table_station = groupes[attributs_station].agg(lambda s: s.value_counts().index[0]).reset_index()
table_station[["latitude", "longitude"]] = (
    table_station["position"].str.split("|", expand=True).astype(float).to_numpy())
table_station["id_amenageur"] = table_station["nom_amenageur"].map(id_acteur)
table_station["id_operateur"] = table_station["nom_operateur"].map(id_acteur)
table_station = table_station[[
    "id_station", "nom_station", "adresse_station", "implantation_station", "condition_acces",
    "latitude", "longitude", "code_insee", "id_amenageur", "id_operateur"]]
journaliser("6_stations", len(points), len(table_station), desaccord.sum(),
            "stations ; concernées = stations dont un attribut prend la valeur la plus fréquente")

# --- Points de charge ----------------------------------------------
prises = ["prise_type_2", "prise_type_combo_ccs", "prise_type_chademo", "prise_type_ef"]
table_point = points[["id_pdc", "puissance_kw", *prises, "date_mise_en_service", "id_station"]].copy()
table_point[prises] = table_point[prises].eq("True").astype(int)
journaliser("7_points", len(points), len(table_point), table_point["date_mise_en_service"].isna().sum(),
            "points de charge ; concernées = points sans date de mise en service")


[1_lecture_points] 166,430 -> 166,430 lignes | concernées : 0 | points de charge du jeu propre
[1_lecture_communes] 34,875 -> 34,875 lignes | concernées : 0 | communes du jeu propre
[2_commune_population_nulle] 34,875 -> 34,876 lignes | concernées : 1 | commune des points ajoutée avec sa population réelle de 0 : Beaumont-en-Verdunois
[3_regions] 22 -> 22 lignes | concernées : 0 | régions
[3_departements] 105 -> 105 lignes | concernées : 0 | départements
[4_acteurs] 4,856 -> 4,856 lignes | concernées : 63,581 | acteurs distincts ; concernées = points dont l'aménageur est aussi l'opérateur
[5_stations_scindees] 166,430 -> 166,430 lignes | concernées : 438 | points de 35 stations réparties sur plusieurs communes : une station par commune
[6_stations] 166,430 -> 49,236 lignes | concernées : 675 | stations ; concernées = stations dont un attribut prend la valeur la plus fréquente
[7_points] 166,430 -> 166,430 lignes | concernées : 65,202 | points de charge ; concernées = points sans date de

In [3]:
# ===================================================================
# CRÉATION DE LA BASE SQLITE, CHARGEMENT ET CONTRÔLES D'INTÉGRITÉ
# ===================================================================
CHEMIN_BASE = Path("../data/database/irve_database.db")
CHEMIN_BASE.parent.mkdir(parents=True, exist_ok=True)

SCHEMA = """
DROP VIEW IF EXISTS v_equipement_commune;
DROP TABLE IF EXISTS POINT_DE_CHARGE;
DROP TABLE IF EXISTS STATION;
DROP TABLE IF EXISTS ACTEUR;
DROP TABLE IF EXISTS COMMUNE;
DROP TABLE IF EXISTS DEPARTEMENT;
DROP TABLE IF EXISTS REGION;

CREATE TABLE REGION (
    code_region      TEXT PRIMARY KEY,
    nom_region       TEXT NOT NULL
);

CREATE TABLE DEPARTEMENT (
    code_departement TEXT PRIMARY KEY,
    nom_departement  TEXT NOT NULL,
    code_region      TEXT NOT NULL REFERENCES REGION (code_region)
);

CREATE TABLE COMMUNE (
    code_insee        TEXT PRIMARY KEY,
    nom_commune       TEXT NOT NULL,
    population        INTEGER NOT NULL CHECK (population >= 0),
    latitude          REAL NOT NULL CHECK (latitude BETWEEN -90 AND 90),
    longitude         REAL NOT NULL CHECK (longitude BETWEEN -180 AND 180),
    distance_pdc_km   REAL NOT NULL CHECK (distance_pdc_km >= 0),
    distance_150kw_km REAL NOT NULL CHECK (distance_150kw_km >= 0),
    desert_recharge   INTEGER NOT NULL CHECK (desert_recharge IN (0, 1)),
    desert_150kw      INTEGER NOT NULL CHECK (desert_150kw IN (0, 1)),
    code_departement  TEXT NOT NULL REFERENCES DEPARTEMENT (code_departement)
);

CREATE TABLE ACTEUR (
    id_acteur  INTEGER PRIMARY KEY,
    nom_acteur TEXT NOT NULL UNIQUE
);

CREATE TABLE STATION (
    id_station           TEXT PRIMARY KEY,
    nom_station          TEXT NOT NULL,
    adresse_station      TEXT NOT NULL,
    implantation_station TEXT NOT NULL,
    condition_acces      TEXT NOT NULL,
    latitude             REAL NOT NULL CHECK (latitude BETWEEN -90 AND 90),
    longitude            REAL NOT NULL CHECK (longitude BETWEEN -180 AND 180),
    code_insee           TEXT NOT NULL REFERENCES COMMUNE (code_insee),
    id_amenageur         INTEGER NOT NULL REFERENCES ACTEUR (id_acteur),
    id_operateur         INTEGER NOT NULL REFERENCES ACTEUR (id_acteur)
);

CREATE TABLE POINT_DE_CHARGE (
    id_pdc               TEXT PRIMARY KEY,
    puissance_kw         REAL NOT NULL CHECK (puissance_kw > 0 AND puissance_kw <= 600),
    prise_type_2         INTEGER NOT NULL CHECK (prise_type_2 IN (0, 1)),
    prise_type_combo_ccs INTEGER NOT NULL CHECK (prise_type_combo_ccs IN (0, 1)),
    prise_type_chademo   INTEGER NOT NULL CHECK (prise_type_chademo IN (0, 1)),
    prise_type_ef        INTEGER NOT NULL CHECK (prise_type_ef IN (0, 1)),
    date_mise_en_service TEXT,
    id_station           TEXT NOT NULL REFERENCES STATION (id_station)
);

CREATE INDEX idx_departement_region ON DEPARTEMENT (code_region);
CREATE INDEX idx_commune_departement ON COMMUNE (code_departement);
CREATE INDEX idx_station_commune ON STATION (code_insee);
CREATE INDEX idx_station_amenageur ON STATION (id_amenageur);
CREATE INDEX idx_station_operateur ON STATION (id_operateur);
CREATE INDEX idx_point_station ON POINT_DE_CHARGE (id_station);

CREATE VIEW v_equipement_commune AS
SELECT c.code_insee,
       c.nom_commune,
       c.population,
       COUNT(DISTINCT s.id_station) AS nb_stations,
       COUNT(p.id_pdc) AS nb_pdc,
       SUM(CASE WHEN p.puissance_kw >= 150 THEN 1 ELSE 0 END) AS nb_pdc_150kw
FROM COMMUNE c
LEFT JOIN STATION s ON s.code_insee = c.code_insee
LEFT JOIN POINT_DE_CHARGE p ON p.id_station = s.id_station
GROUP BY c.code_insee, c.nom_commune, c.population;
"""

tables = {
    "REGION": table_region, "DEPARTEMENT": table_departement, "COMMUNE": table_commune,
    "ACTEUR": table_acteur, "STATION": table_station, "POINT_DE_CHARGE": table_point,
}

with sqlite3.connect(CHEMIN_BASE) as connexion:
  connexion.execute("PRAGMA foreign_keys = ON")
  connexion.executescript(SCHEMA)
  for nom, donnees in tables.items():
    donnees.to_sql(nom, connexion, if_exists="append", index=False)

  # Contrôles d'intégrité
  violations = connexion.execute("PRAGMA foreign_key_check").fetchall()
  if violations:
    raise ValueError(f"Clés étrangères invalides : {violations[:5]}")
  bilan = []
  for nom, donnees in tables.items():
    en_base = connexion.execute(f"SELECT COUNT(*) FROM {nom}").fetchone()[0]
    if en_base != len(donnees):
      raise ValueError(f"{nom} : {en_base} lignes en base pour {len(donnees)} préparées")
    bilan.append({"table": nom, "lignes": en_base})
  equipement = pd.read_sql("SELECT SUM(nb_pdc) AS points, SUM(nb_stations) AS stations,"
                           " SUM(nb_pdc_150kw) AS points_150kw FROM v_equipement_commune",
                           connexion)
connexion.close()

print(pd.DataFrame(bilan).to_string(index=False))
print("\nClés étrangères : aucune violation")
print("Vue v_equipement_commune :", equipement.to_dict("records")[0])
print(f"Base : {CHEMIN_BASE.resolve()} ({CHEMIN_BASE.stat().st_size / 1e6:.1f} Mo)")


          table  lignes
         REGION      22
    DEPARTEMENT     105
        COMMUNE   34876
         ACTEUR    4856
        STATION   49236
POINT_DE_CHARGE  166430

Clés étrangères : aucune violation
Vue v_equipement_commune : {'points': 166430, 'stations': 49236, 'points_150kw': 30918}
Base : C:\Users\cgboh\OneDrive\Desktop\irve-bornes-recharge-ml_Charmelle\data\database\irve_database.db (32.8 Mo)


## Exemples de requêtes

Les requêtes ci-dessous répondent directement à la problématique : où se situent les déserts de recharge, et quelle part de la population ils concernent.


In [4]:
with sqlite3.connect(CHEMIN_BASE) as connexion:
  deserts_par_region = pd.read_sql("""
      SELECT r.nom_region,
             COUNT(*) AS communes,
             SUM(c.desert_recharge) AS deserts_10km,
             SUM(c.desert_150kw) AS deserts_150kw_30km,
             SUM(CASE WHEN c.desert_recharge = 1 THEN c.population ELSE 0 END) AS habitants_desert_10km
      FROM COMMUNE c
      JOIN DEPARTEMENT d ON d.code_departement = c.code_departement
      JOIN REGION r ON r.code_region = d.code_region
      GROUP BY r.nom_region
      ORDER BY deserts_10km DESC""", connexion)
  equipement_departement = pd.read_sql("""
      SELECT d.code_departement, d.nom_departement,
             SUM(v.population) AS population,
             SUM(v.nb_pdc) AS nb_pdc,
             ROUND(SUM(v.nb_pdc) * 10000.0 / SUM(v.population), 1) AS pdc_pour_10000_hab
      FROM v_equipement_commune v
      JOIN COMMUNE c ON c.code_insee = v.code_insee
      JOIN DEPARTEMENT d ON d.code_departement = c.code_departement
      GROUP BY d.code_departement, d.nom_departement
      ORDER BY pdc_pour_10000_hab""", connexion)
connexion.close()

print("Déserts de recharge par région")
print(deserts_par_region.to_string(index=False))
print("\nDépartements les moins équipés (points pour 10 000 habitants)")
print(equipement_departement.head(10).to_string(index=False))


Déserts de recharge par région
                nom_region  communes  deserts_10km  deserts_150kw_30km  habitants_desert_10km
   Bourgogne-Franche-Comté      3685           165                   0                  28393
                 Grand Est      5110           136                   6                  21448
                 Normandie      2644            93                   0                  46391
           Hauts-de-France      3782            68                   0                  16142
      Auvergne-Rhône-Alpes      4024            44                  37                  10454
        Nouvelle-Aquitaine      4293            37                  87                  12632
                 Occitanie      4446            22                   6                   3280
                     Corse       360            21                   3                   4569
                    Guyane        22            19                  22                 162255
Provence-Alpes-Côte d'Azur   